# Try the AI dog repeller in your browser

No Raspberry Pi, camera or installation needed: everything runs here on Google's computers.
Run the cells from top to bottom (**Runtime → Run all**). Takes about 3 minutes.

In [ ]:
REPO_URL = "https://github.com/mazenelnaghy-code/dog-detection"  # <-- the project's GitHub page
!git clone -q $REPO_URL repo
%cd repo
!pip install -q ultralytics && pip install -q --no-deps -e .

## 1. Watch the full system work
Children and a street cat walk past (nothing must happen), then an Egyptian street dog walks up to
the camera: the buzzer beeps, the ultrasonic repeller fires in bursts, then the system calms down.
Hardware is simulated; watch the log and the video.

In [ ]:
!python -m dog_repeller --demo --no-window --no-dashboard --set camera.loop=false --record demo.avi
!python tools/make_gif.py demo.avi demo.gif --width 480 --every 2
from IPython.display import Image
Image(filename="demo.gif")

## 2. Score the model on the sample photos (10 dogs, 5 cats/children)

In [ ]:
!python tools/evaluate.py --dogs samples/dogs --no-dogs samples/no_dogs --variant none night far

## 3. Try YOUR photo
Upload any photo; dogs are boxed in red/green, and the danger zone is shown in yellow.

In [ ]:
from google.colab import files
import cv2, sys
from IPython.display import Image, display
sys.path.insert(0, "src")
from dog_repeller.config import load_config
from dog_repeller.detector import DogDetector
from dog_repeller.zone import DangerZone, frame_level
from dog_repeller.overlay import draw_overlay

cfg = load_config("config/default.yaml")
detector, zone = DogDetector(cfg.model), DangerZone(cfg.zone)
for name in files.upload():
    img = cv2.imread(name)
    h, w = img.shape[:2]
    threats = zone.assess(detector.detect(img), w, h)
    status = {"level": frame_level(threats).name, "state": "photo", "armed": True, "fps": 0}
    cv2.imwrite("result.jpg", draw_overlay(img, zone.pixel_rect(w, h), threats, status))
    print(f"{name}: {len(threats)} dog(s), level {status['level']}")
    display(Image(filename="result.jpg", width=640))

## Next steps
* Build the real device: see the README (Raspberry Pi setup, wiring, parts list).
* Test with your own videos: `python -m dog_repeller --source my_video.mp4 --sim --set camera.realtime=true`